# Pipeline ML v6 — Dimensionnement opérateurs par pôle
## Clients S et W — Random Forest


In [19]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from scipy import stats
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.1f}'.format)

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

# 8 pôles — GLOBAL séparé en GLOBAL_S et GLOBAL_W
POLES       = ['BULK_S','BULK_W','PICKING_S','PICKING_W',
               'PROMO_S','PROMO_W','GLOBAL_S','GLOBAL_W']
RANDOM_SEED = 42

# Productivités réelles (OL/heure × 7h/jour)
# Source : document Productivités OETP
HEURES_PAR_JOUR = 7
PRODUCTIVITE_HORAIRE = {
    'BULK_S'    : 26.5,
    'BULK_W'    : 26.5,
    'PICKING_S' : 157.0,
    'PICKING_W' : 157.0,
    'PROMO_S'   : 125.0,
    'PROMO_W'   : 125.0,
    'GLOBAL_S'  : 27.6,
    'GLOBAL_W'  : 27.6,
}
PRODUCTIVITE = {p: round(v * HEURES_PAR_JOUR) for p, v in PRODUCTIVITE_HORAIRE.items()}

print('Imports OK — Random Forest | 8 pôles')
print('\nProductivités journalières (OL/op/jour) :')
for p, v in PRODUCTIVITE.items():
    print(f'  {p:12s} : {PRODUCTIVITE_HORAIRE[p]:5.1f} OL/h × {HEURES_PAR_JOUR}h = {v:>5} OL/op/jour')


Imports OK — Random Forest | 8 pôles

Productivités journalières (OL/op/jour) :
  BULK_S       :  26.5 OL/h × 7h =   186 OL/op/jour
  BULK_W       :  26.5 OL/h × 7h =   186 OL/op/jour
  PICKING_S    : 157.0 OL/h × 7h =  1099 OL/op/jour
  PICKING_W    : 157.0 OL/h × 7h =  1099 OL/op/jour
  PROMO_S      : 125.0 OL/h × 7h =   875 OL/op/jour
  PROMO_W      : 125.0 OL/h × 7h =   875 OL/op/jour
  GLOBAL_S     :  27.6 OL/h × 7h =   193 OL/op/jour
  GLOBAL_W     :  27.6 OL/h × 7h =   193 OL/op/jour


## 1. Chargement des données

In [20]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
DATA_FOLDER      = Path('../DATA/Input')
SHEET_NAME       = 'Grid Results'
TRANSPORT_FILE   = Path('../DATA/Input/20260319 Plan de transport simplifié pour le Crunch.xlsx')
PREVISIONS_FILE  = Path('../DATA/Input/Prévisions S_W.xlsx')
# ─────────────────────────────────────────────────────────────────────────────

def load_all_files(folder, sheet):
    files = sorted(folder.glob('20*.xlsx'))
    if not files:
        raise FileNotFoundError(f"Aucun fichier trouvé dans {folder}")
    frames = []
    for f in files:
        try:
            xls = pd.ExcelFile(f)
        except Exception as e:
            warnings.warn(f"Impossible de lire {f.name} : {e}")
            continue
        if sheet not in xls.sheet_names:
            warnings.warn(f"Fichier ignoré : {f.name} — onglet '{sheet}' absent")
            continue
        df = pd.read_excel(f, sheet_name=sheet, header=0)
        df.columns = df.columns.str.strip().str.upper()
        frames.append(df)
        print(f"  ✓ {f.name} — {len(df):,} lignes")
    raw = pd.concat(frames, ignore_index=True)
    raw['DATE'] = pd.to_datetime(raw['DATE'])
    return raw.sort_values('DATE').drop_duplicates()

# ── Données historiques ───────────────────────────────────────────────────────
print(f"Chargement depuis : {DATA_FOLDER.resolve()}")
raw = load_all_files(DATA_FOLDER, SHEET_NAME)
print(f"Total : {len(raw):,} lignes | {raw['DATE'].min().date()} → {raw['DATE'].max().date()}")

# ── Plan de transport : jours actifs par (client, type_op, jour_semaine) ──────
transport = pd.read_excel(TRANSPORT_FILE, sheet_name='Plan de transport', header=0)
transport.columns = ['CLIENT','TERRITOIRE','WORKFLOW','LUNDI','MARDI','MERCREDI','JEUDI','VENDREDI']
transport = transport.dropna(subset=['WORKFLOW'])
transport['CLIENT']    = transport['CLIENT'].ffill()
transport['TERRITOIRE']= transport['TERRITOIRE'].ffill()

DAYS_MAP = {'LUNDI':0,'MARDI':1,'MERCREDI':2,'JEUDI':3,'VENDREDI':4}
DAYS     = list(DAYS_MAP.keys())
transport['jours_list'] = transport[DAYS].apply(
    lambda r: [DAYS_MAP[d] for d in DAYS if r[d]=='X'], axis=1)

# Dict workflow → jours actifs (utilisé pour les features)
WORKFLOW_JOURS = dict(zip(transport['WORKFLOW'], transport['jours_list']))

# Nb pays actifs par (client, type_op, dow) → feature clé
records = []
for _, row in transport.iterrows():
    m = re.match(r'PREPARATION_([SW])_(BULK|PICKING|PROMO|GLOBAL)', row['WORKFLOW'])
    if not m:
        continue
    cli, top = m.group(1), m.group(2)
    for jour, dow in DAYS_MAP.items():
        records.append({'client': cli, 'type_op': top, 'dow': dow,
                        'actif': 1 if row[jour]=='X' else 0})

TRANSPORT_FEAT = (pd.DataFrame(records)
                  .groupby(['client','type_op','dow'])['actif']
                  .sum()
                  .reset_index()
                  .rename(columns={'actif':'nb_pays_actifs'}))

print(f"\nPlan de transport chargé : {len(transport)} workflows")
print(TRANSPORT_FEAT.pivot_table(index=['client','type_op'], columns='dow',
                                  values='nb_pays_actifs').to_string())

# ── Prévisions mensuelles S/W (cadrage 2026) ──────────────────────────────────
prev_raw = pd.read_excel(PREVISIONS_FILE, sheet_name='Feuil1', header=0)
prev_raw.columns = ['DATE','WORKFLOW','QUANTITY','_3','_4','_5','NOTE']
PREVISIONS_SW = (prev_raw[['DATE','WORKFLOW','QUANTITY']]
                 .dropna(subset=['WORKFLOW','QUANTITY'])
                 .assign(DATE=lambda d: pd.to_datetime(d['DATE']),
                         CLIENT=lambda d: d['WORKFLOW'].str.extract(r'PREPARATION_([SW])_')[0],
                         ZONE=lambda d: d['WORKFLOW'].str.replace(r'PREPARATION_[SW]_','',regex=True)))
print(f"\nPrévisions S/W chargées : {len(PREVISIONS_SW)} lignes")
print(f"Période : {PREVISIONS_SW['DATE'].min().date()} → {PREVISIONS_SW['DATE'].max().date()}")


Chargement depuis : C:\Users\Rodri\Desktop\UTT\P26\crunch\DATA\Input
  ✓ 2023.xlsx — 13,583 lignes
  ✓ 2024.xlsx — 19,413 lignes
  ✓ 2025.xlsx — 18,341 lignes
  ✓ 2026.xlsx — 2,958 lignes
Total : 54,295 lignes | 2023-01-05 → 2026-02-27

Plan de transport chargé : 74 workflows
dow               0   1   2    3   4
client type_op                      
S      BULK     5.0 7.0 3.0  6.0 6.0
       PICKING  5.0 7.0 3.0  6.0 6.0
       PROMO    5.0 7.0 3.0  6.0 6.0
W      BULK    15.0 6.0 7.0 13.0 4.0
       PICKING 16.0 6.0 7.0 14.0 4.0
       PROMO   16.0 6.0 7.0 14.0 4.0

Prévisions S/W chargées : 165 lignes
Période : 2025-10-01 → 2026-12-01


## 2. Parsing et agrégation S/W

In [21]:
def parse_workflow(wf):
    """Extrait (client, type_op, pays) depuis un code WORKFLOW."""
    m = re.match(r'PREPARATION_([SW])_(BULK|PICKING|GLOBAL|PROMO)(?:_(.+))?', wf)
    if m:
        return m.group(1), m.group(2), m.group(3) or 'ALL'
    return None, None, None

parsed = raw['WORKFLOW'].apply(lambda x: pd.Series(parse_workflow(x),
                                                    index=['CLIENT','TYPE_OP','PAYS']))
raw = pd.concat([raw, parsed], axis=1)

print("Clients :")
print(raw['CLIENT'].value_counts().to_dict())
print("\nTypes d'opération :")
print(raw['TYPE_OP'].value_counts().to_dict())
print(f"\nNombre de pays : {raw['PAYS'].nunique()}")


Clients :
{'W': 33108, 'S': 21187}

Types d'opération :
{'PICKING': 18706, 'BULK': 17203, 'PROMO': 16874, 'GLOBAL': 1512}

Nombre de pays : 66


In [22]:
# ── Agrégation par (DATE, CLIENT, TYPE_OP) ───────────────────────────────────
# GLOBAL_S et GLOBAL_W sont maintenant des pôles distincts (8 pôles au total)

# Tous les types d'opération séparés par client
daily_raw = (
    raw.groupby(['DATE','CLIENT','TYPE_OP'])['QUANTITY']
    .sum()
    .unstack(['CLIENT','TYPE_OP'], fill_value=0)
    .reset_index()
)
# Aplatir les colonnes multi-niveaux : (S, BULK) → BULK_S, (W, GLOBAL) → GLOBAL_W
daily_raw.columns = (['DATE'] +
    [f'{top}_{cli}' for cli, top in daily_raw.columns[1:]])
daily_raw.columns.name = None

# S'assurer que tous les pôles existent
for p in POLES:
    if p not in daily_raw.columns:
        daily_raw[p] = 0

# Nb pays actifs ce jour (global et par client)
nb_pays = raw.groupby('DATE')['PAYS'].nunique().rename('NB_PAYS')
daily   = daily_raw.merge(nb_pays, on='DATE', how='left')
for cli in ['S','W']:
    nb = raw[raw['CLIENT']==cli].groupby('DATE')['PAYS'].nunique().rename(f'NB_PAYS_{cli}')
    daily = daily.merge(nb, on='DATE', how='left')

# ── Réindexation sur calendrier continu ───────────────────────────────────────
date_range = pd.date_range(daily['DATE'].min(), daily['DATE'].max(), freq='D')
daily = daily.set_index('DATE').reindex(date_range).rename_axis('DATE').reset_index()
for col in POLES + ['NB_PAYS','NB_PAYS_S','NB_PAYS_W']:
    if col in daily.columns:
        daily[col] = daily[col].fillna(0)

daily['TOTAL']   = daily[POLES].sum(axis=1)
daily['TOTAL_S'] = daily[[p for p in POLES if p.endswith('_S')]].sum(axis=1)
daily['TOTAL_W'] = daily[[p for p in POLES if p.endswith('_W')]].sum(axis=1)
daily['IS_WEEKEND'] = (daily['DATE'].dt.dayofweek >= 5).astype(int)

print(f"Jours calendaires : {len(daily)}")
print(f"Dont weekends     : {daily['IS_WEEKEND'].sum()}")
print(f"Jours ouvrés      : {(daily['IS_WEEKEND']==0).sum()}")
print("\nStats par pôle :")
display(daily[POLES].describe().round(0))


Jours calendaires : 1150
Dont weekends     : 328
Jours ouvrés      : 822

Stats par pôle :


,BULK_S,BULK_W,PICKING_S,PICKING_W,PROMO_S,PROMO_W,GLOBAL_S,GLOBAL_W
count,1150.0,1150.0,1150.0,1150.0,1150.0,1150.0,1150.0,1150.0
mean,365.0,376.0,4717.0,6283.0,1896.0,3784.0,6978.0,10442.0
std,406.0,376.0,5205.0,6498.0,2422.0,4037.0,7612.0,10144.0
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,282.0,355.0,3690.0,5894.0,1246.0,3310.0,5682.0,9950.0
75%,560.0,586.0,7067.0,9826.0,2806.0,6956.0,10396.0,17310.0
max,3089.0,3070.0,31196.0,54532.0,15810.0,17787.0,40587.0,65742.0


## 3. Feature Engineering + Statistiques de référence

In [23]:
# ── Jours fériés français ────────────────────────────────────────────────────
FERIES_FR = pd.to_datetime([
    '2023-01-01','2023-04-10','2023-05-01','2023-05-08','2023-05-18','2023-05-29',
    '2023-07-14','2023-08-15','2023-11-01','2023-11-11','2023-12-25',
    '2024-01-01','2024-04-01','2024-05-01','2024-05-08','2024-05-09','2024-05-20',
    '2024-07-14','2024-08-15','2024-11-01','2024-11-11','2024-12-25',
    '2025-01-01','2025-04-21','2025-05-01','2025-05-08','2025-05-29','2025-06-09',
    '2025-07-14','2025-08-15','2025-11-01','2025-11-11','2025-12-25',
    '2026-01-01','2026-04-06','2026-05-01','2026-05-08','2026-05-14','2026-05-25',
    '2026-07-14','2026-08-15','2026-11-01','2026-11-11','2026-12-25',
])
# UK : pas de départ J-1 (donc pas de prépa J-1 → impacte W uniquement)
FERIES_UK_VEILLE = FERIES_FR  # même jours fériés UK → on bloque J-1

def get_transport_feature(client, type_op, dow, transport_feat):
    """Retourne le nb de pays actifs ce (client, type_op, jour_semaine)."""
    mask = ((transport_feat['client']  == client) &
            (transport_feat['type_op'] == type_op) &
            (transport_feat['dow']     == dow))
    rows = transport_feat[mask]
    return int(rows['nb_pays_actifs'].iloc[0]) if len(rows) else 0

def build_features(df, poles, transport_feat,
                   lags=[1,7,14,28,365], windows=[7,14,28,90]):
    out = df.copy().sort_values('DATE').reset_index(drop=True)

    # ── Features temporelles ─────────────────────────────────────────────────
    out['DOW']         = out['DATE'].dt.dayofweek
    out['WEEK']        = out['DATE'].dt.isocalendar().week.astype(int)
    out['MONTH']       = out['DATE'].dt.month
    out['QUARTER']     = out['DATE'].dt.quarter
    out['DAY_OF_YEAR'] = out['DATE'].dt.dayofyear
    out['YEAR']        = out['DATE'].dt.year
    out['IS_FERIE']    = out['DATE'].isin(FERIES_FR).astype(int)
    out['IS_VEILLE_FERIE'] = out['DATE'].apply(
        lambda d: (d + pd.Timedelta(days=1)) in FERIES_FR).astype(int)
    out['IS_WEEKEND']  = (out['DATE'].dt.dayofweek >= 5).astype(int)
    out['DOW_SIN']     = np.sin(2*np.pi*out['DOW']/7)
    out['DOW_COS']     = np.cos(2*np.pi*out['DOW']/7)
    out['MONTH_SIN']   = np.sin(2*np.pi*out['MONTH']/12)
    out['MONTH_COS']   = np.cos(2*np.pi*out['MONTH']/12)

    # ── Pics commerciaux ─────────────────────────────────────────────────────
    out['BLACK_FRIDAY'] = ((out['MONTH']==11) & (out['DATE'].dt.day>=20)).astype(int)
    out['NOEL_PEAK']    = ((out['MONTH']==12) & (out['DATE'].dt.day<=24)).astype(int)
    out['SOLDES']       = (((out['MONTH']==1)) |
                           ((out['MONTH']==6) & (out['DATE'].dt.day>=24))).astype(int)

    # ── Transitions entre collections ────────────────────────────────────────
    out['TRANS_W2S'] = (out['MONTH'].isin([3,4])).astype(int)
    out['TRANS_S2W'] = (out['MONTH'].isin([8,9])).astype(int)
    out['WEEKS_TO_TRANS'] = out['DATE'].apply(lambda d: min(
        abs((d - pd.Timestamp(f'{d.year}-04-01')).days),
        abs((d - pd.Timestamp(f'{d.year}-09-01')).days)) // 7)

    # ── Contraintes transport S/W par jour de semaine ─────────────────────────
    # Pour chaque (client, type_op) : nb de pays actifs CE jour → signal de charge
    for cli in ['S','W']:
        for top in ['BULK','PICKING','PROMO']:
            col = f'TRANSPORT_{cli}_{top}_ACTIF'
            out[col] = out['DOW'].apply(
                lambda d: get_transport_feature(cli, top, d, transport_feat))

    # Règle UK (client W) : J-1 avant férié UK → pas de prépa W_UK
    out['UK_VEILLE_FERIE'] = out['DATE'].apply(
        lambda d: int((d + pd.Timedelta(days=1)) in FERIES_UK_VEILLE)).astype(int)

    # ── Lags, rolling et features dérivées — par pôle ────────────────────────
    for pole in poles:
        for lag in lags:
            out[f'{pole}_LAG{lag}'] = out[pole].shift(lag)
        for w in windows:
            out[f'{pole}_ROLL{w}'] = out[pole].shift(1).rolling(w, min_periods=1).mean()
        # Tendance long terme — correction biais sur-prédiction
        out[f'{pole}_TREND90'] = out[pole].shift(1).rolling(90, min_periods=14).mean()
        # Momentum semaine/semaine
        roll7  = out[pole].shift(1).rolling(7,  min_periods=1).mean()
        roll14 = out[pole].shift(8).rolling(7,  min_periods=1).mean()
        out[f'{pole}_WOW'] = (roll7 / (roll14 + 1e-6)).clip(0, 5)

    # ── Features BULK et GLOBAL spécifiques ─────────────────────────────────────
    for cli in ['S','W']:
        pick = f'PICKING_{cli}'
        if pick in poles:
            out[f'PICKING_{cli}_LAG1_for_BULK'] = out[pick].shift(1)
        bulk = f'BULK_{cli}'
        if bulk in poles:
            out[f'BULK_{cli}_RATIO_TOTAL'] = (
                out[bulk].shift(1) / (out['TOTAL'].shift(1) + 1e-6)).clip(0,1)
            out[f'BULK_{cli}_STD14'] = (
                out[bulk].shift(1).rolling(14, min_periods=3).std().fillna(0))

    # ── NB_PAYS et TOTAL ─────────────────────────────────────────────────────
    for cli in ['S','W','']:
        suffix = f'_{cli}' if cli else ''
        col = f'NB_PAYS{suffix}'
        if col in out.columns:
            out[f'NB_PAYS{suffix}_LAG1']  = out[col].shift(1)
            out[f'NB_PAYS{suffix}_ROLL7'] = out[col].shift(1).rolling(7, min_periods=1).mean()

    for lag in [1,7]:
        out[f'TOTAL_LAG{lag}']   = out['TOTAL'].shift(lag)
        out[f'TOTAL_S_LAG{lag}'] = out['TOTAL_S'].shift(lag)
        out[f'TOTAL_W_LAG{lag}'] = out['TOTAL_W'].shift(lag)

    # ── Cibles J+1 ───────────────────────────────────────────────────────────
    for pole in poles:
        out[f'TARGET_{pole}'] = out[pole].shift(-1)

    return out

features_df = build_features(daily, POLES, TRANSPORT_FEAT)

FEATURE_COLS = [c for c in features_df.columns
                if c not in (['DATE'] + POLES +
                              ['TOTAL','TOTAL_S','TOTAL_W','NB_PAYS','NB_PAYS_S','NB_PAYS_W','IS_WEEKEND'])
                and not c.startswith('TARGET_')]

print(f"Features construites : {len(FEATURE_COLS)}")


Features construites : 131


In [24]:
# ── Statistiques de référence pour le Z-score (calculées une seule fois) ──────
# IMPORTANT : calculées sur `daily` (données brutes agrégées) et non sur
# `features_df` (qui contient des jours réindexés à 0 faussant le sigma)

def compute_reference_stats(df_daily, poles, feries):
    """
    Calcule mu et sigma historiques par pôle.
    Base : jours ouvrés non fériés avec volume > 0 (jours d'activité réelle).
    """
    stats_ref = {}
    for pole in poles:
        if pole not in df_daily.columns:
            stats_ref[pole] = {'mu': 1.0, 'sigma': 1.0, 'q1': 0.0, 'q3': 1.0}
            continue
        hist = df_daily[
            (df_daily['IS_WEEKEND'] == 0) &
            (~df_daily['DATE'].isin(feries)) &
            (df_daily[pole] > 0)          # ← jours actifs uniquement
        ][pole]
        if len(hist) < 5:
            stats_ref[pole] = {'mu': 1.0, 'sigma': 1.0, 'q1': 0.0, 'q3': 1.0}
            continue
        stats_ref[pole] = {
            'mu'   : float(hist.mean()),
            'sigma': float(hist.std()) if hist.std() > 0 else 1.0,
            'q1'   : float(hist.quantile(0.25)),
            'q3'   : float(hist.quantile(0.75)),
        }
    return stats_ref

def zscore(value, pole, stats_ref):
    """Calcule le Z-score d'une valeur pour un pôle donné."""
    mu    = stats_ref[pole]['mu']
    sigma = stats_ref[pole]['sigma']
    return round((value - mu) / sigma, 2)

# Calcul sur `daily` — données brutes sans jours réindexés à 0
STATS_REF = compute_reference_stats(features_df, POLES, FERIES_FR)

# Vérification : la date max des données doit couvrir au moins 2 ans
date_max = daily['DATE'].max()
date_min = daily['DATE'].min()
nb_jours = (date_max - date_min).days
if nb_jours < 365:
    print(f"⚠ ATTENTION : données sur seulement {nb_jours} jours — Z-scores peu fiables.")
    print("  Chargez tous les fichiers 2023-2025 avant de lancer les warnings.")
else:
    print(f"✓ Base Z-score : {nb_jours} jours ({date_min.date()} → {date_max.date()})")

print("\nStatistiques de référence par pôle (base Z-score, jours actifs uniquement) :")
print(f"{'Pôle':12s} {'Moyenne':>10} {'Écart-type':>12} {'Q1':>8} {'Q3':>8} {'N jours':>8}")
print("-" * 64)
for pole in POLES:
    s = STATS_REF[pole]
    n = len(daily[
        (daily['IS_WEEKEND'] == 0) &
        (~daily['DATE'].isin(FERIES_FR)) &
        (daily[pole] > 0)
    ])
    print(f"{pole:12s} {s['mu']:>10.0f} {s['sigma']:>12.0f} {s['q1']:>8.0f} {s['q3']:>8.0f} {n:>8}")


✓ Base Z-score : 1149 jours (2023-01-05 → 2026-02-27)

Statistiques de référence par pôle (base Z-score, jours actifs uniquement) :
Pôle            Moyenne   Écart-type       Q1       Q3  N jours
----------------------------------------------------------------
BULK_S              554          382      286      703      757
BULK_W              574          319      366      730      752
PICKING_S          7201         4848     3822     9510      753
PICKING_W          9634         5696     5987    12042      750
PROMO_S            3374         2335     1688     4473      646
PROMO_W            5969         3559     3717     8199      729
GLOBAL_S          10596         7049     5833    14025      757
GLOBAL_W          15969         8311    10354    20559      752


## 4. Walk-Forward Validation

In [25]:
def walk_forward_split(df, train_start, test_start, step_days=1):
    """Expanding window — retourne des DATES pour éviter les désalignements d'index."""
    active = df[
        (df['IS_WEEKEND'] == 0) &
        (~df['DATE'].isin(FERIES_FR))
    ].copy().reset_index(drop=True)

    dates   = active['DATE'].values
    t_start = pd.Timestamp(test_start)
    t_end   = dates.max()
    current = t_start

    while current <= t_end:
        tr_mask = dates < current
        te_mask = dates == current
        if tr_mask.sum() > 50 and te_mask.sum() > 0:
            yield (active.loc[tr_mask, 'DATE'].values,
                   active.loc[te_mask, 'DATE'].values)
        current += pd.Timedelta(days=step_days)


TRAIN_START = '2023-01-01'
TEST_START  = '2026-01-01'   # 3 ans de train (2023+2024+2025), test sur 2026
STEP_DAYS   = 1

splits = list(walk_forward_split(
    features_df.dropna(subset=[f'TARGET_{POLES[0]}']),
    TRAIN_START, TEST_START, STEP_DAYS))
print(f"Splits walk-forward (jours ouvrés, 3 ans de train) : {len(splits)}")
if splits:
    print(f"  Premier split — train : {len(splits[0][0])} jours")
    print(f"  Dernière date de test : {pd.Timestamp(max(splits[-1][1])).date()}")


Splits walk-forward (jours ouvrés, 3 ans de train) : 40
  Premier split — train : 751 jours
  Dernière date de test : 2026-02-26


## 5. Entraînement Random Forest

In [26]:
def tune_rf(df, pole, feature_cols, n_iter=40):
    """
    RandomizedSearchCV sur RandomForestRegressor.
    Plus rapide qu'Optuna pour RF car l'espace est plus restreint.
    """
    target_col = f'TARGET_{pole}'
    data = df[
        (df['IS_WEEKEND'] == 0) &
        (~df['DATE'].isin(FERIES_FR))
    ].dropna(subset=[target_col] + feature_cols).copy()

    cut    = max(30, int(len(data) * 0.8))
    X_tr, y_tr = data.iloc[:cut][feature_cols], data.iloc[:cut][target_col]

    param_dist = {
        'n_estimators'     : [200, 300, 500, 800],
        'max_depth'        : [None, 6, 10, 15, 20],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf' : [1, 2, 4, 8],
        'max_features'     : ['sqrt', 'log2', 0.5, 0.7],
        'bootstrap'        : [True],
    }

    rf = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
    search = RandomizedSearchCV(rf, param_dist, n_iter=n_iter,
                                cv=3, scoring='neg_mean_absolute_error',
                                random_state=RANDOM_SEED, n_jobs=-1)
    search.fit(X_tr, y_tr)
    mae_val = -search.best_score_
    print(f"  {pole} — MAE val: {mae_val:.0f} | params: {search.best_params_}")
    return search.best_params_


def predict_quantile_rf(model, X, quantile=0.9):
    """
    Prédit les intervalles de confiance depuis les arbres individuels du RF.
    Retourne (prédiction moyenne, borne basse q10, borne haute q90).
    """
    # Prédictions de chaque arbre individuellement
    tree_preds = np.array([tree.predict(X) for tree in model.estimators_])
    # tree_preds shape: (n_estimators, n_samples)
    pred_mean = np.maximum(0, tree_preds.mean(axis=0))
    pred_low  = np.maximum(0, np.percentile(tree_preds, (1-quantile)*100, axis=0))
    pred_high = np.maximum(0, np.percentile(tree_preds, quantile*100,     axis=0))
    return pred_mean, pred_low, pred_high


def train_evaluate_pole_rf(df, pole, feature_cols, splits, rf_params):
    """Walk-forward Random Forest — retourne prédictions + intervalles."""
    target_col = f'TARGET_{pole}'
    data = df.dropna(subset=[target_col] + feature_cols).copy()

    oof_mean, oof_low, oof_high = [], [], []
    oof_actual, oof_dates = [], []

    for tr_dates, te_dates in splits:
        X_tr = data[data['DATE'].isin(tr_dates)][feature_cols]
        y_tr = data[data['DATE'].isin(tr_dates)][target_col]
        X_te = data[data['DATE'].isin(te_dates)][feature_cols]
        y_te = data[data['DATE'].isin(te_dates)][target_col]
        if len(X_tr) < 30 or len(X_te) == 0:
            continue

        model = RandomForestRegressor(**rf_params, random_state=RANDOM_SEED, n_jobs=-1)
        model.fit(X_tr, y_tr)

        mean, low, high = predict_quantile_rf(model, X_te)
        oof_mean.extend(mean)
        oof_low.extend(low)
        oof_high.extend(high)
        oof_actual.extend(y_te.values)
        oof_dates.extend(te_dates)

    # Modèle final sur toutes les données
    final = RandomForestRegressor(**rf_params, random_state=RANDOM_SEED, n_jobs=-1)
    final.fit(data[feature_cols], data[target_col])

    df_oos = pd.DataFrame({
        'date'  : oof_dates,
        'actual': oof_actual,
        'pred'  : oof_mean,
        'low'   : oof_low,
        'high'  : oof_high,
    }).sort_values('date').reset_index(drop=True)

    return final, df_oos


# ── Tuning + entraînement ─────────────────────────────────────────────────────
print("Tuning Random Forest (40 itérations par pôle)...")
BEST_PARAMS_RF = {}
for pole in POLES:
    print(f"  → {pole}...", end=' ', flush=True)
    BEST_PARAMS_RF[pole] = tune_rf(features_df, pole, FEATURE_COLS)

print("\nEntraînement walk-forward (7 pôles)...")
models  = {}
results = {}
for pole in POLES:
    print(f"  → {pole}...", end=' ')
    m, res = train_evaluate_pole_rf(features_df, pole, FEATURE_COLS,
                                    splits, BEST_PARAMS_RF[pole])
    models[pole]  = m
    results[pole] = res
    print(f"{len(res)} points OOS")

print("\nTerminé.")


Tuning Random Forest (40 itérations par pôle)...
  → BULK_S...   BULK_S — MAE val: 164 | params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_features': 0.7, 'max_depth': None, 'bootstrap': True}
  → BULK_W...   BULK_W — MAE val: 202 | params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_features': 0.7, 'max_depth': None, 'bootstrap': True}
  → PICKING_S...   PICKING_S — MAE val: 2294 | params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 6, 'bootstrap': True}
  → PICKING_W...   PICKING_W — MAE val: 2377 | params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 0.5, 'max_depth': None, 'bootstrap': True}
  → PROMO_S...   PROMO_S — MAE val: 676 | params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.7, 'max_depth': 6, 'bootstrap': True}
  → PROMO_W...   PROMO_W — MAE val: 1182 | params: {'n_estim

## 6. Métriques

In [27]:
def evaluate(res):
    y_true, y_pred = res['actual'], res['pred']
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    mask = y_true > 0
    mape = (np.abs((y_true[mask]-y_pred[mask])/y_true[mask]).mean()*100) if mask.sum() else np.nan
    coverage = ((res['actual'] >= res['low']) & (res['actual'] <= res['high'])).mean() * 100
    return {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape, 'Couverture IC (%)': coverage}

# Ajout Z-score sur les prédictions OOS
def add_zscore_to_oos(results, poles, stats_ref):
    """Ajoute une colonne z_score_pred et z_score_actual à chaque DataFrame OOS."""
    for pole in poles:
        res = results[pole]
        if res.empty:
            continue
        mu    = stats_ref[pole]['mu']
        sigma = stats_ref[pole]['sigma']
        res['z_pred']   = ((res['pred']   - mu) / sigma).round(2)
        res['z_actual'] = ((res['actual'] - mu) / sigma).round(2)
    return results

results = add_zscore_to_oos(results, POLES, STATS_REF)

metrics_df = pd.DataFrame({p: evaluate(results[p]) for p in POLES}).T.round(2)
print("── Métriques Random Forest ──")
display(metrics_df)

for cli in ['S','W']:
    poles_cli = [p for p in POLES if p.endswith(f'_{cli}')]
    mets = metrics_df.loc[poles_cli]
    print(f"\nClient {cli} — R² moyen : {mets['R²'].mean():.2f} "
          f"| MAPE moyen : {mets['MAPE (%)'].mean():.1f}%"
          f"| Couverture IC : {mets['Couverture IC (%)'].mean():.1f}%")

# Afficher les 10 prédictions OOS avec Z-score les plus élevés (toutes pôles)
print("\n── Top 10 prédictions OOS avec Z-score le plus élevé ──")
all_oos = pd.concat([
    results[p][['date','actual','pred','z_pred','z_actual','low','high']].assign(pole=p)
    for p in POLES if not results[p].empty
])
all_oos['z_abs'] = all_oos['z_pred'].abs()
display(all_oos.nlargest(10, 'z_abs')[['date','pole','actual','pred','z_actual','z_pred','low','high']].reset_index(drop=True))


── Métriques Random Forest ──


,MAE,RMSE,R²,MAPE (%),Couverture IC (%)
BULK_S,223.5,355.5,0.7,37.9,90.0
BULK_W,140.1,240.5,0.5,26.1,85.0
PICKING_S,1393.4,2061.4,0.8,28.1,62.5
PICKING_W,1550.0,2066.2,0.7,25.8,80.0
PROMO_S,948.4,1499.3,0.8,28.2,65.0
PROMO_W,948.4,1510.1,0.8,21.3,82.5
GLOBAL_S,2164.3,3338.5,0.8,23.5,65.0
GLOBAL_W,2175.5,3225.6,0.8,18.7,82.5



Client S — R² moyen : 0.76 | MAPE moyen : 29.4%| Couverture IC : 70.6%

Client W — R² moyen : 0.71 | MAPE moyen : 23.0%| Couverture IC : 82.5%

── Top 10 prédictions OOS avec Z-score le plus élevé ──


,date,pole,actual,pred,z_actual,z_pred,low,high
0,2026-02-02,BULK_S,1404.0,1467.1,2.2,2.4,870.4,1906.0
1,2026-01-15,BULK_S,1218.0,1420.9,1.7,2.3,926.8,1827.8
2,2026-02-12,BULK_S,796.0,1379.4,0.6,2.2,1011.9,1780.3
3,2026-02-09,BULK_S,1508.0,1328.2,2.5,2.0,917.2,1775.1
4,2026-02-16,PROMO_S,6689.0,7987.4,1.4,2.0,5964.0,10342.9
5,2026-01-02,GLOBAL_W,0.0,0.0,-1.9,-1.9,0.0,0.0
6,2026-01-09,GLOBAL_W,0.0,0.0,-1.9,-1.9,0.0,0.0
7,2026-01-16,GLOBAL_W,0.0,0.0,-1.9,-1.9,0.0,0.0
8,2026-01-23,GLOBAL_W,0.0,0.0,-1.9,-1.9,0.0,0.0
9,2026-01-30,GLOBAL_W,0.0,0.0,-1.9,-1.9,0.0,0.0


## 7. Conversion volume → opérateurs et prédiction J+1

In [28]:
# ── Productivités réelles (déjà définies dans les imports, rappel ici) ────────
# Source : document Productivités OETP
# Formule : OL/heure × 7 heures/jour
print("Productivités utilisées pour la conversion volume → opérateurs :")
for p, v in PRODUCTIVITE.items():
    print(f"  {p:12s} : {PRODUCTIVITE_HORAIRE[p]:5.1f} OL/h × {HEURES_PAR_JOUR}h = {v:>5} OL/op/jour")

def volume_to_operators(volumes, productivite, min_ops=1):
    """Convertit un dict {pole: volume} en {pole: nb_operateurs}."""
    return {p: max(min_ops, int(np.ceil(v / productivite[p])))
            for p, v in volumes.items()}

# Vérification : exemple sur un jour type
print("\nExemple de conversion pour un jour type :")
exemple = {'BULK_S':500,'BULK_W':400,'PICKING_S':15000,'PICKING_W':12000,
           'PROMO_S':8000,'PROMO_W':6000,'GLOBAL_S':3000,'GLOBAL_W':2500}
ops = volume_to_operators(exemple, PRODUCTIVITE)
print(f"{'Pôle':12s} {'Volume':>10} {'Productivité':>14} {'Opérateurs':>12}")
print("-" * 52)
for p in POLES:
    print(f"{p:12s} {exemple[p]:>10,} {PRODUCTIVITE[p]:>14,} {ops[p]:>12}")
print(f"{'TOTAL':12s} {sum(exemple.values()):>10,} {'':>14} {sum(ops.values()):>12}")


Productivités utilisées pour la conversion volume → opérateurs :
  BULK_S       :  26.5 OL/h × 7h =   186 OL/op/jour
  BULK_W       :  26.5 OL/h × 7h =   186 OL/op/jour
  PICKING_S    : 157.0 OL/h × 7h =  1099 OL/op/jour
  PICKING_W    : 157.0 OL/h × 7h =  1099 OL/op/jour
  PROMO_S      : 125.0 OL/h × 7h =   875 OL/op/jour
  PROMO_W      : 125.0 OL/h × 7h =   875 OL/op/jour
  GLOBAL_S     :  27.6 OL/h × 7h =   193 OL/op/jour
  GLOBAL_W     :  27.6 OL/h × 7h =   193 OL/op/jour

Exemple de conversion pour un jour type :
Pôle             Volume   Productivité   Opérateurs
----------------------------------------------------
BULK_S              500            186            3
BULK_W              400            186            3
PICKING_S        15,000          1,099           14
PICKING_W        12,000          1,099           11
PROMO_S           8,000            875           10
PROMO_W           6,000            875            7
GLOBAL_S          3,000            193           16
GLOBAL_

## 8. Replay walk-forward

In [29]:
def replay_walk_forward(features_df, results, poles, productivite, stats_ref):
    """Reconstruit le tableau réel vs prédit avec intervalles + Z-score par pôle."""
    rows = []
    for pole in poles:
        mu    = stats_ref[pole]['mu']
        sigma = stats_ref[pole]['sigma']
        for _, r in results[pole].iterrows():
            prod = productivite[pole]
            rows.append({
                'date'     : r['date'],
                'pole'     : pole,
                'client'   : pole.split('_')[-1] if '_' in pole else 'S+W',
                'vol_reel' : r['actual'],
                'vol_pred' : r['pred'],
                'vol_low'  : r['low'],
                'vol_high' : r['high'],
                'z_pred'   : round((r['pred']   - mu) / sigma, 2),
                'z_actual' : round((r['actual'] - mu) / sigma, 2),
                'ops_reel' : max(1, int(np.ceil(r['actual'] / prod))),
                'ops_pred' : max(1, int(np.ceil(r['pred']   / prod))),
                'ops_low'  : max(1, int(np.ceil(r['low']    / prod))),
                'ops_high' : max(1, int(np.ceil(r['high']   / prod))),
            })
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values(['date','pole'])

replay = replay_walk_forward(features_df, results, POLES, PRODUCTIVITE, STATS_REF)
print(f"Jours simulés : {replay['date'].nunique()}")
display(replay.head(16))


Jours simulés : 40


,date,pole,client,vol_reel,vol_pred,vol_low,vol_high,z_pred,z_actual,ops_reel,ops_pred,ops_low,ops_high
0,2026-01-02,BULK_S,S,0.0,8.8,0.0,4.2,-1.4,-1.4,1,1,1,1
40,2026-01-02,BULK_W,W,0.0,0.0,0.0,0.0,-1.8,-1.8,1,1,1,1
240,2026-01-02,GLOBAL_S,S,0.0,124.3,0.0,0.0,-1.5,-1.5,1,1,1,1
280,2026-01-02,GLOBAL_W,W,0.0,0.0,0.0,0.0,-1.9,-1.9,1,1,1,1
80,2026-01-02,PICKING_S,S,0.0,103.6,0.0,0.0,-1.5,-1.5,1,1,1,1
120,2026-01-02,PICKING_W,W,0.0,134.4,0.0,0.0,-1.7,-1.7,1,1,1,1
160,2026-01-02,PROMO_S,S,0.0,1.9,0.0,0.0,-1.4,-1.4,1,1,1,1
200,2026-01-02,PROMO_W,W,0.0,0.0,0.0,0.0,-1.7,-1.7,1,1,1,1
1,2026-01-05,BULK_S,S,638.0,928.8,463.4,1325.0,1.0,0.2,4,5,3,8
41,2026-01-05,BULK_W,W,677.0,426.6,214.3,622.6,-0.5,0.3,4,3,2,4


In [30]:
# ── Agrégats par jour (nécessaires pour l'export) ────────────────────────────
daily_ops   = replay.groupby('date')[['ops_reel','ops_pred','ops_low','ops_high']].sum()
daily_ops_s = replay[replay['client']=='S'].groupby('date')[['ops_reel','ops_pred','ops_low','ops_high']].sum()
daily_ops_w = replay[replay['client']=='W'].groupby('date')[['ops_reel','ops_pred','ops_low','ops_high']].sum()

ecart = (daily_ops['ops_pred'] - daily_ops['ops_reel']).abs().mean()
couv  = ((daily_ops['ops_reel'] >= daily_ops['ops_low']) &
         (daily_ops['ops_reel'] <= daily_ops['ops_high'])).mean() * 100
print(f"Écart moyen absolu total : {ecart:.1f} opérateur(s)/jour")
print(f"Couverture IC 80%        : {couv:.1f}%")


Écart moyen absolu total : 18.0 opérateur(s)/jour
Couverture IC 80%        : 85.0%


## 9. Analyse du risque de sous-staffing et sur-staffing

Pour chaque journée de la période de test, on quantifie deux risques opérationnels :

- **Sous-staffing** : `ops_pred < ops_reel` → manque d'opérateurs → risque de retard
- **Sur-staffing**  : `ops_pred > ops_reel` → excès d'opérateurs → coût inutile

Les **intervalles de confiance P10/P90** du Random Forest mesurent l'incertitude :
- `risque_sous_staff = ops_high − ops_pred` — opérateurs supplémentaires qui pourraient manquer
- `risque_sur_staff  = ops_pred − ops_low`  — opérateurs excédentaires possibles
- **Niveau de risque** : FAIBLE (amplitude IC < 15%) · MODÉRÉ (15–30%) · ÉLEVÉ (> 30%)


In [31]:
def classify_risk(amplitude_ratio):
    """Classifie le niveau de risque selon l'amplitude relative de l'IC P10/P90."""
    if amplitude_ratio < 0.15:
        return 'FAIBLE'
    elif amplitude_ratio < 0.30:
        return 'MODÉRÉ'
    else:
        return 'ÉLEVÉ'


def analyze_staffing_risk(replay_df, poles):
    """
    Analyse le risque réalisé sur la période de test (walk-forward OOS).
    Compare ops_pred vs ops_reel et calcule l'exposition IC par pôle et par jour.
    - SOUS-STAFF : sous-prédiction → manque d'opérateurs
    - SUR-STAFF  : sur-prédiction  → excès d'opérateurs
    """
    rows = []
    for _, r in replay_df.iterrows():
        ecart           = r['ops_pred'] - r['ops_reel']
        ecart_abs       = abs(ecart)
        ecart_pct       = ecart_abs / (r['ops_reel'] + 1e-6) * 100
        amplitude       = r['ops_high'] - r['ops_low']
        amplitude_ratio = amplitude / (r['ops_pred'] + 1e-6)

        if ecart < -0.5:
            statut = 'SOUS-STAFF'
        elif ecart > 0.5:
            statut = 'SUR-STAFF'
        else:
            statut = 'OK'

        rows.append({
            'date'              : r['date'],
            'pole'              : r['pole'],
            'client'            : r['client'],
            'ops_reel'          : r['ops_reel'],
            'ops_pred'          : r['ops_pred'],
            'ops_low'           : r['ops_low'],
            'ops_high'          : r['ops_high'],
            'ecart_ops'         : round(ecart, 1),
            'ecart_abs'         : round(ecart_abs, 1),
            'ecart_pct'         : round(ecart_pct, 1),
            'statut_reel'       : statut,
            'risque_sous_staff' : max(0, r['ops_high'] - r['ops_pred']),
            'risque_sur_staff'  : max(0, r['ops_pred'] - r['ops_low']),
            'amplitude_ic'      : amplitude,
            'amplitude_ratio'   : round(amplitude_ratio, 3),
            'niveau_risque'     : classify_risk(amplitude_ratio),
        })
    return pd.DataFrame(rows)


risk_historique = analyze_staffing_risk(replay, POLES)
print(f"Période analysée : {pd.to_datetime(risk_historique['date']).min().date()} → "
      f"{pd.to_datetime(risk_historique['date']).max().date()}")
print(f"Observations (jours × pôles) : {len(risk_historique)}")
print(f"\nRépartition des statuts réalisés :")
display(risk_historique['statut_reel'].value_counts().to_frame())

# ── Résumé par pôle ───────────────────────────────────────────────────────────
resume_risque = risk_historique.groupby('pole').agg(
    jours_total     = ('date',              'nunique'),
    pct_sous_staff  = ('statut_reel',       lambda x: round((x == 'SOUS-STAFF').mean()*100, 1)),
    pct_sur_staff   = ('statut_reel',       lambda x: round((x == 'SUR-STAFF').mean()*100, 1)),
    pct_ok          = ('statut_reel',       lambda x: round((x == 'OK').mean()*100, 1)),
    ecart_moy_abs   = ('ecart_abs',         'mean'),
    ecart_max       = ('ecart_abs',         'max'),
    risque_sous_moy = ('risque_sous_staff', 'mean'),
    risque_sur_moy  = ('risque_sur_staff',  'mean'),
    niveau_dominant = ('niveau_risque',     lambda x: x.mode()[0] if len(x) else 'N/A'),
).round(1)
print("\n── Résumé du risque par pôle (période de test) ──")
display(resume_risque)


Période analysée : 2026-01-02 → 2026-02-26
Observations (jours × pôles) : 320

Répartition des statuts réalisés :


,count
statut_reel,
OK,121
SUR-STAFF,113
SOUS-STAFF,86



── Résumé du risque par pôle (période de test) ──


,jours_total,pct_sous_staff,pct_sur_staff,pct_ok,ecart_moy_abs,ecart_max,risque_sous_moy,risque_sur_moy,niveau_dominant
pole,,,,,,,,,
BULK_S,40,27.5,32.5,40.0,1.1,8,2.0,1.8,ÉLEVÉ
BULK_W,40,15.0,20.0,65.0,0.6,6,1.4,1.2,ÉLEVÉ
GLOBAL_S,40,37.5,42.5,20.0,11.2,63,15.5,13.9,ÉLEVÉ
GLOBAL_W,40,32.5,47.5,20.0,11.3,60,17.4,16.1,ÉLEVÉ
PICKING_S,40,20.0,32.5,47.5,1.2,5,1.6,1.6,ÉLEVÉ
PICKING_W,40,30.0,40.0,30.0,1.4,5,2.1,1.8,ÉLEVÉ
PROMO_S,40,35.0,22.5,42.5,1.0,7,1.1,1.1,ÉLEVÉ
PROMO_W,40,17.5,45.0,37.5,1.2,7,2.0,1.8,ÉLEVÉ


In [32]:
# ── Export Excel — Analyse des risques (basé sur OOS walk-forward) ────────────
RISK_FILE = 'staffing_risk_oos.xlsx'

colors_sev = {
    'ÉLEVÉ'   : PatternFill('solid', fgColor='F7C1C1'),
    'MODÉRÉ'  : PatternFill('solid', fgColor='FAC775'),
    'FAIBLE'  : PatternFill('solid', fgColor='C0DD97'),
    'SOUS-STAFF' : PatternFill('solid', fgColor='F7C1C1'),
    'SUR-STAFF'  : PatternFill('solid', fgColor='C6D9F1'),
    'OK'         : PatternFill('solid', fgColor='C0DD97'),
}

def style_risk_sheet(ws, risk_col):
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    for i in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(i)].width = 16
    col_names = [c.value for c in ws[1]]
    if risk_col in col_names:
        idx = col_names.index(risk_col)
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            val = row[idx].value
            fc  = colors_sev.get(val)
            if fc:
                for cell in row:
                    cell.fill = fc

# Jours à risque ÉLEVÉ uniquement
jours_eleves = risk_historique[risk_historique['niveau_risque'] == 'ÉLEVÉ'].copy()

with pd.ExcelWriter(RISK_FILE, engine='openpyxl') as writer:
    risk_historique.to_excel(writer, sheet_name='Risque_OOS', index=False)
    resume_risque.to_excel(writer, sheet_name='Résumé_pôle')
    jours_eleves.to_excel(writer, sheet_name='Jours_risque_élevé', index=False)

    style_risk_sheet(writer.sheets['Risque_OOS'],          'statut_reel')
    style_risk_sheet(writer.sheets['Jours_risque_élevé'],  'niveau_risque')
    for sheet in ['Résumé_pôle']:
        for cell in writer.sheets[sheet][1]:
            cell.font = Font(bold=True)

print(f"Exporté : {RISK_FILE}")
print(f"  Onglets :")
print(f"    Risque_OOS          — {len(risk_historique):,} lignes (période de test)")
print(f"    Résumé_pôle         — stats par pôle")
print(f"    Jours_risque_élevé  — {len(jours_eleves):,} lignes (risque ÉLEVÉ uniquement)")

# Résumé rapide par statut
total   = len(risk_historique)
sous    = (risk_historique['statut_reel'] == 'SOUS-STAFF').sum()
sur     = (risk_historique['statut_reel'] == 'SUR-STAFF').sum()
ok      = (risk_historique['statut_reel'] == 'OK').sum()
print(f"\n  SOUS-STAFF : {sous:4d} ({sous/total*100:.1f}%)")
print(f"  SUR-STAFF  : {sur:4d} ({sur/total*100:.1f}%)")
print(f"  OK         : {ok:4d} ({ok/total*100:.1f}%)")


Exporté : staffing_risk_oos.xlsx
  Onglets :
    Risque_OOS          — 320 lignes (période de test)
    Résumé_pôle         — stats par pôle
    Jours_risque_élevé  — 237 lignes (risque ÉLEVÉ uniquement)

  SOUS-STAFF :   86 (26.9%)
  SUR-STAFF  :  113 (35.3%)
  OK         :  121 (37.8%)


## 9. Export résultats

In [33]:
OUTPUT_FILE = 'resultats_previsions_rf.xlsx'

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    metrics_df.to_excel(writer, sheet_name='Métriques')
    for pole in POLES:
        results[pole].to_excel(writer, sheet_name=f'OOS_{pole[:10]}', index=False)
    replay.to_excel(writer, sheet_name='Replay_opérateurs', index=False)
    daily_ops.to_excel(writer, sheet_name='Total_jour')
    daily_ops_s.to_excel(writer, sheet_name='Total_jour_S')
    daily_ops_w.to_excel(writer, sheet_name='Total_jour_W')

    if 'risk_historique' in dir():
        risk_historique.to_excel(writer, sheet_name='Risque_staffing', index=False)
        resume_risque.to_excel(writer, sheet_name='Résumé_risque_pôle', index=False)

print(f"Exporté : {OUTPUT_FILE}")


Exporté : resultats_previsions_rf.xlsx


## 10. Détection des valeurs aberrantes — fichier warnings

In [34]:
def detect_anomalies_from_results(results, stats_ref, poles):
    """
    Calcule les Z-scores et détecte les anomalies directement depuis
    les prédictions OOS (results[pole]) — sans passer par predict_full_year.

    Pour chaque prédiction OOS :
      Z = (volume_prédit - moyenne_historique) / écart_type_historique

    Sévérité :
      FAIBLE   : 0.5 ≤ |Z| < 1
      MOYEN    : 1 ≤ |Z| < 1.5
      ÉLEVÉ    : 1.5 ≤ |Z| < 2
      CRITIQUE : |Z| ≥ 2

    Direction :
      SURCHARGE   : Z > 0 → volume élevé → risque sous-effectif
      SOUS-CHARGE : Z < 0 → volume faible → sur-effectif
    """
    warn_list = []

    for pole in poles:
        res = results[pole]
        if res.empty:
            continue

        mu    = stats_ref[pole]['mu']
        sigma = stats_ref[pole]['sigma']

        for _, row in res.iterrows():
            val = float(row['pred'])
            if val == 0:
                continue   # volume nul = artefact contrainte transport

            z    = round((val - mu) / sigma, 2)
            absz = abs(z)

            if   absz >= 2.0: sev = 'CRITIQUE'
            elif absz >= 1.5: sev = 'ÉLEVÉ'
            elif absz >= 1.0: sev = 'MOYEN'
            elif absz >= 0.5: sev = 'FAIBLE'
            else: continue

            direction = 'SURCHARGE' if z > 0 else 'SOUS-CHARGE'
            ecart_pct = round((val - mu) / mu * 100, 1) if mu > 0 else 0
            date_val  = pd.Timestamp(row['date'])

            warn_list.append({
                'Date'          : date_val.strftime('%Y-%m-%d'),
                'Jour'          : date_val.strftime('%A'),
                'Pôle'          : pole,
                'Client'        : pole.split('_')[-1] if '_' in pole else 'S+W',
                'Volume prédit' : int(val),
                'Volume réel'   : int(row['actual']),
                'Moyenne hist.' : round(mu, 0),
                'Écart (%)'     : ecart_pct,
                'Z-score'       : z,
                'IC low'        : int(row['low']),
                'IC high'       : int(row['high']),
                'Dans IC'       : int(row['actual'] >= row['low'] and row['actual'] <= row['high']),
                'Direction'     : direction,
                'Sévérité'      : sev,
                'Raison'        : f'{direction} — Z={z:.2f} ({ecart_pct:+.1f}% vs moy. {mu:.0f})',
            })

    df_w = pd.DataFrame(warn_list)
    if not df_w.empty:
        sev_order = {'CRITIQUE': 0, 'ÉLEVÉ': 1, 'MOYEN': 2, 'FAIBLE': 3}
        df_w['_ord'] = df_w['Sévérité'].map(sev_order)
        df_w = (df_w.sort_values(['_ord', 'Z-score'], ascending=[True, False])
                    .drop('_ord', axis=1)
                    .reset_index(drop=True))

    return df_w


df_warnings = detect_anomalies_from_results(results, STATS_REF, POLES)

print(f"Anomalies détectées sur les prédictions OOS : {len(df_warnings)}")
if not df_warnings.empty:
    print("\nRépartition par sévérité :")
    print(df_warnings['Sévérité'].value_counts()
          .reindex(['CRITIQUE','ÉLEVÉ','MOYEN','FAIBLE']).fillna(0).astype(int).to_string())
    print("\nRépartition par direction :")
    print(df_warnings['Direction'].value_counts().to_string())
    display(df_warnings.head(20))
else:
    print("Aucune anomalie détectée.")
    print("  → Vérifiez que STATS_REF est calculé sur les 3 ans complets (2023-2025).")


Anomalies détectées sur les prédictions OOS : 126

Répartition par sévérité :
Sévérité
CRITIQUE     4
ÉLEVÉ       21
MOYEN       36
FAIBLE      65

Répartition par direction :
Direction
SOUS-CHARGE    70
SURCHARGE      56


,Date,Jour,Pôle,Client,Volume prédit,Volume réel,Moyenne hist.,Écart (%),Z-score,IC low,IC high,Dans IC,Direction,Sévérité,Raison
0,2026-02-02,Monday,BULK_S,S,1467,1404,554.0,164.9,2.4,870,1905,1,SURCHARGE,CRITIQUE,SURCHARGE — Z=2.39 (+164.9% vs moy. 554)
1,2026-01-15,Thursday,BULK_S,S,1420,1218,554.0,156.5,2.3,926,1827,1,SURCHARGE,CRITIQUE,SURCHARGE — Z=2.27 (+156.5% vs moy. 554)
2,2026-02-12,Thursday,BULK_S,S,1379,796,554.0,149.0,2.2,1011,1780,0,SURCHARGE,CRITIQUE,SURCHARGE — Z=2.16 (+149.0% vs moy. 554)
3,2026-02-09,Monday,BULK_S,S,1328,1508,554.0,139.8,2.0,917,1775,1,SURCHARGE,CRITIQUE,SURCHARGE — Z=2.03 (+139.8% vs moy. 554)
4,2026-02-16,Monday,PROMO_S,S,7987,6689,3374.0,136.7,2.0,5964,10342,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.98 (+136.7% vs moy. 3374)
5,2026-01-29,Thursday,BULK_S,S,1272,1407,554.0,129.7,1.9,730,1768,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.88 (+129.7% vs moy. 554)
6,2026-02-02,Monday,GLOBAL_S,S,23699,26261,10596.0,123.7,1.9,15381,31599,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.86 (+123.7% vs moy. 10596)
7,2026-01-26,Monday,BULK_S,S,1257,1833,554.0,127.0,1.8,679,1840,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.84 (+127.0% vs moy. 554)
8,2026-01-22,Thursday,BULK_S,S,1253,1262,554.0,126.3,1.8,697,1795,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.83 (+126.3% vs moy. 554)
9,2026-02-05,Thursday,BULK_S,S,1253,1281,554.0,126.4,1.8,762,1783,1,SURCHARGE,ÉLEVÉ,SURCHARGE — Z=1.83 (+126.4% vs moy. 554)


In [35]:
WARNINGS_FILE = 'warnings_predictions_oos.xlsx'

fill_map = {
    'CRITIQUE': PatternFill('solid', fgColor='E8201A'),
    'ÉLEVÉ'   : PatternFill('solid', fgColor='F7C1C1'),
    'MOYEN'   : PatternFill('solid', fgColor='FAC775'),
    'FAIBLE'  : PatternFill('solid', fgColor='FFF2CC'),
}
font_map = {
    'CRITIQUE': Font(bold=True, color='FFFFFF'),
    'ÉLEVÉ'   : Font(bold=True),
    'MOYEN'   : Font(bold=False),
    'FAIBLE'  : Font(bold=False),
}
col_widths = {
    'Date':14,'Jour':12,'Pôle':12,'Client':8,
    'Volume prédit':15,'Volume réel':14,'Moyenne hist.':15,
    'Écart (%)':12,'Z-score':10,'IC low':10,'IC high':10,
    'Dans IC':9,'Direction':14,'Sévérité':12,'Raison':55,
}

with pd.ExcelWriter(WARNINGS_FILE, engine='openpyxl') as writer:

    # Onglet principal Warnings
    if df_warnings.empty:
        pd.DataFrame([{'Message':'Aucune anomalie détectée'}]).to_excel(
            writer, sheet_name='Warnings', index=False)
    else:
        df_warnings.to_excel(writer, sheet_name='Warnings', index=False)
        ws = writer.sheets['Warnings']
        sev_idx = list(df_warnings.columns).index('Sévérité')
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            sev = row[sev_idx].value
            for cell in row:
                if fill_map.get(sev): cell.fill = fill_map[sev]
            row[sev_idx].font = font_map.get(sev, Font())
        for i, col in enumerate(df_warnings.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = col_widths.get(col, 14)
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center')

    # Onglet Z-scores complets sur toutes les prédictions OOS
    all_oos = pd.concat([
        results[p][['date','actual','pred','low','high']].assign(
            pole=p,
            z_pred  = lambda d, p=p: ((d['pred']   - STATS_REF[p]['mu']) / STATS_REF[p]['sigma']).round(2),
            z_actual= lambda d, p=p: ((d['actual'] - STATS_REF[p]['mu']) / STATS_REF[p]['sigma']).round(2),
            dans_ic = lambda d: ((d['actual'] >= d['low']) & (d['actual'] <= d['high'])).astype(int),
        )
        for p in POLES if not results[p].empty
    ]).sort_values(['date','pole']).reset_index(drop=True)
    all_oos['date'] = pd.to_datetime(all_oos['date']).dt.strftime('%Y-%m-%d')
    all_oos.to_excel(writer, sheet_name='Z-scores_OOS', index=False)
    ws_z = writer.sheets['Z-scores_OOS']
    for i in range(1, len(all_oos.columns)+1):
        ws_z.column_dimensions[get_column_letter(i)].width = 14
    for cell in ws_z[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')

    # Onglet résumé métriques
    metrics_df.to_excel(writer, sheet_name='Métriques')

print(f"Exporté : {WARNINGS_FILE}")
print(f"  Onglets : Warnings | Z-scores_OOS | Métriques")


Exporté : warnings_predictions_oos.xlsx
  Onglets : Warnings | Z-scores_OOS | Métriques


## 11. Export PowerBI

In [36]:
from datetime import datetime

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
POWERBI_FOLDER = Path(r'C:\\Users\\Rodri\\Desktop\\UTT\\P26\\crunch\\DATA\\Result')
PREFIX = 'CRUNCH'
# ─────────────────────────────────────────────────────────────────────────────

def export_for_powerbi(results, metrics_df, daily, replay,
                       poles, productivite, folder, prefix):
    """
    Exporte les CSV pour PowerBI Desktop.
    Inclut les intervalles de confiance RF dans toutes les tables.
    """
    folder.mkdir(parents=True, exist_ok=True)
    ts      = datetime.now().strftime('%Y-%m-%d %H:%M')
    exports = {}

    # 1. Prédiction J+1 avec fourchette
    last_date = features_df['DATE'].max()
    j1_rows   = []
    for pole in poles:
        row_feat = features_df[features_df['DATE'] == last_date]
        if row_feat.empty: continue
        X = row_feat[FEATURE_COLS]
        mean, low, high = predict_quantile_rf(models[pole], X)
        ops_c = max(1, int(np.ceil(mean[0] / productivite[pole])))
        ops_l = max(1, int(np.ceil(low[0]  / productivite[pole])))
        ops_h = max(1, int(np.ceil(high[0] / productivite[pole])))
        j1_rows.append({
            'date_prediction' : last_date.strftime('%Y-%m-%d'),
            'date_j1'         : (last_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
            'pole'            : pole,
            'client'          : pole.split('_')[-1] if '_' in pole else 'S+W',
            'type_operation'  : pole.split('_')[0]  if '_' in pole else 'GLOBAL',
            'volume_predit'   : round(mean[0]),
            'nb_operateurs'   : ops_c,
            'ops_min_p10'     : ops_l,
            'ops_max_p90'     : ops_h,
            'horodatage'      : ts,
        })
    exports['predictions_j1'] = pd.DataFrame(j1_rows)


    # 4. Historique OOS avec intervalles
    oos_rows = []
    for pole in poles:
        res = results[pole].copy()
        res['pole']           = pole
        res['client']         = pole.split('_')[-1] if '_' in pole else 'S+W'
        res['type_operation'] = pole.split('_')[0]  if '_' in pole else 'GLOBAL'
        res['erreur_abs']     = (res['actual'] - res['pred']).abs()
        res['erreur_pct']     = (res['erreur_abs'] / (res['actual'] + 1e-6) * 100).round(1)
        res['dans_ic']        = ((res['actual'] >= res['low']) &
                                  (res['actual'] <= res['high'])).astype(int)
        res['date']           = pd.to_datetime(res['date']).dt.strftime('%Y-%m-%d')
        res['horodatage']     = ts
        oos_rows.append(res)
    exports['historique_oos'] = pd.concat(oos_rows, ignore_index=True)

    # 5. Métriques
    df_met = metrics_df.copy().reset_index().rename(columns={'index':'pole'})
    df_met['client']         = df_met['pole'].apply(lambda p: p.split('_')[-1] if '_' in p else 'S+W')
    df_met['type_operation'] = df_met['pole'].apply(lambda p: p.split('_')[0]  if '_' in p else 'GLOBAL')
    df_met['horodatage']     = ts
    exports['metriques_modeles'] = df_met

    # 6. Opérateurs réel vs prédit avec fourchette
    df_ops = replay.copy()
    df_ops['date']       = pd.to_datetime(df_ops['date']).dt.strftime('%Y-%m-%d')
    df_ops['horodatage'] = ts
    exports['operateurs_jour'] = df_ops

    # Écriture CSV
    print(f"Export PowerBI → {folder}")
    for name, df_exp in exports.items():
        path = folder / f'{prefix}_{name}.csv'
        df_exp.to_csv(path, index=False, encoding='utf-8-sig', sep=';')
        print(f"  ✓ {path.name} — {len(df_exp):,} lignes")

    # ── Copie des fichiers xlsx vers le dossier PowerBI ──────────────────────
    xlsx_files = {
        'resultats_previsions_rf.xlsx' : 'Résultats prévisions RF (OOS + métriques)',
        'warnings_predictions_oos.xlsx': 'Warnings + Z-scores OOS',
        'staffing_risk_oos.xlsx'       : 'Analyse sous/sur-staffing OOS',
    }
    for xlsx_name, desc in xlsx_files.items():
        src_path = Path(xlsx_name)
        dst_path = folder / xlsx_name
        if src_path.exists():
            import shutil
            shutil.copy2(src_path, dst_path)
            print(f"  ✓ {xlsx_name} — {desc}")
        else:
            print(f"  ⚠ {xlsx_name} introuvable — lancez d'abord les cellules d'export §10 et §12")

    # Métadonnées
    meta = pd.DataFrame([{
        'derniere_maj'   : ts,
        'modele'         : 'Random Forest',
        'nb_poles'       : len(poles),
        'date_min_oos'   : exports['historique_oos']['date'].min(),
        'date_max_oos'   : exports['historique_oos']['date'].max(),

        'couverture_ic'  : f"{exports['historique_oos']['dans_ic'].mean()*100:.1f}%",
        'nb_warnings'    : len(df_warnings) if 'df_warnings' in dir() else 'N/A',
        'fichiers_xlsx'  : 'resultats_previsions_rf.xlsx, warnings_predictions_2026.xlsx',
    }])
    meta.to_csv(folder / f'{PREFIX}_metadata.csv', index=False, encoding='utf-8-sig', sep=';')
    print(f"  ✓ {PREFIX}_metadata.csv")


try:
    export_for_powerbi(results, metrics_df, daily, replay,
                       POLES, PRODUCTIVITE, POWERBI_FOLDER, PREFIX)
    print(f"\n{'='*55}\n  Export terminé → {POWERBI_FOLDER}\n{'='*55}")
except Exception as e:
    print(f"\n⚠ Export échoué : {e}")
    print(f"  Vérifiez que POWERBI_FOLDER est accessible : {POWERBI_FOLDER}")


Export PowerBI → C:\Users\Rodri\Desktop\UTT\P26\crunch\DATA\Result
  ✓ CRUNCH_predictions_j1.csv — 8 lignes
  ✓ CRUNCH_historique_oos.csv — 320 lignes
  ✓ CRUNCH_metriques_modeles.csv — 8 lignes
  ✓ CRUNCH_operateurs_jour.csv — 320 lignes
  ✓ resultats_previsions_rf.xlsx — Résultats prévisions RF (OOS + métriques)
  ✓ warnings_predictions_oos.xlsx — Warnings + Z-scores OOS
  ✓ staffing_risk_oos.xlsx — Analyse sous/sur-staffing OOS
  ✓ CRUNCH_metadata.csv

  Export terminé → C:\Users\Rodri\Desktop\UTT\P26\crunch\DATA\Result
